# UDnet ต้นฉบับ (เส้นฐาน) บน Colab

<a href="https://colab.research.google.com/github/nantapongpa-a11y/UDnet/blob/UDnet-Colab/UDnet_Baseline_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

ใช้โค้ด UDnet ของ repo ตรงๆ ไม่แก้วิธี · คู่กับ `UDnetU_Colab.ipynb` (งานใหม่) — ทั้งสองใช้ Drive โฟลเดอร์เดียวกัน
จึงได้ **split เดียวกัน ชุดทดสอบ 90 ภาพเดียวกัน และตารางผลรวมกัน**

| § | ทำอะไร | เวลา (T4) |
|---|---|---|
| 1 | ตั้งค่า + mount Drive | < 1 นาที |
| 2 | clone repo | < 1 นาที |
| 3 | โหลด/แตก UIEB + weight ผู้แต่ง, แบ่ง 800/90, เลือกชุดย่อย | 3–10 นาทีครั้งแรก |
| 4 | ประเมิน weight ผู้แต่ง และภาพดิบ | ~2 นาที |
| 5 | เทรน UDnet ตั้งแต่ epoch 1 (ต่อเองถ้า runtime หลุด) | หลายชั่วโมง |
| 6 | ประเมิน weight ที่เทรนเอง + ตารางรวมทุก run | ~2 นาที |
| 7 | (ทางเลือก) วัด checkpoint ระหว่างทาง — เทรนสั้นกว่านี้พอไหม | ~3 นาที |

**ก่อนรัน:** `Runtime → Change runtime type → T4 GPU`

**สองโหมดการเทรน**
- โหมดเปเปอร์ (`ITER_BUDGET = None`): 500 epoch เหมือนเปเปอร์ ใช้ทำซ้ำผลของผู้แต่ง
- โหมดเส้นโค้งข้อมูล (`ITER_BUDGET = 40000`): คำนวณจำนวน epoch ให้ทุกขนาดชุดฝึก (200/400/800) ได้ **จำนวน iteration เท่ากัน**
  ตามแผนในสไลด์ ใช้ค่าเดียวกับ `ITERS` ใน notebook งานใหม่

**เรื่องรหัสไฟล์ (ตรวจแล้ว):** UIEB บน Google Drive เป็น `.rar` ที่ **เข้ารหัส AES** รหัสเป็นของที่ผู้สร้างชุดข้อมูลเผยแพร่เองบน
[หน้าโครงการ UIEB](https://li-chongyi.github.io/proj_benchmark.html) (ใช้เพื่องานวิชาการที่ไม่ใช่เชิงพาณิชย์) — ใส่ให้แล้วใน `colab_utils.py`
ถ้าแตกไม่ได้หรือจำนวนภาพไม่ครบ notebook จะหยุดพร้อมบอกสาเหตุ · weight ผู้แต่ง `UDnet.pth` ไม่มีรหัส

ถ้าดาวน์โหลดขึ้น **quota exceeded / too many users**: เปิดลิงก์ที่พิมพ์ออกมา ดาวน์โหลดเอง แล้ววางไว้ใน `UDnet_work/downloads/` ด้วยชื่อเดิม แล้วรันเซลล์ใหม่

## §1 ตั้งค่า

In [ ]:
REPO_URL    = "https://github.com/nantapongpa-a11y/UDnet.git"
BRANCH      = "UDnet-Colab"
SEED        = 42     # seed ของ split train/test — ต้องเท่ากันทุก notebook ทุก run
TRAIN_SEED  = 123    # seed ของการเทรน (repo default) — เปลี่ยนตอนทำ 3 seed
N_SUBSET    = 800    # จำนวนภาพเทรน 800 / 400 / 200 (ชุดย่อยซ้อนกันเสมอ: 200 ⊂ 400 ⊂ 800)
BATCH       = 6      # ค่า default ของ repo
N_EPOCHS    = 500    # โหมดเปเปอร์ (ใช้เมื่อ ITER_BUDGET = None) — รอบแรกลอง 2 เพื่อเช็ก pipeline
ITER_BUDGET = None   # โหมดเส้นโค้ง: ใส่ 40000
USE_DRIVE   = True   # เก็บไฟล์ดาวน์โหลด, weight, log, split ลง Google Drive กันหาย

import math
IT_PER_EPOCH = math.ceil(N_SUBSET / BATCH)                # Train.py ไม่ drop batch สุดท้าย
if ITER_BUDGET:
    EPOCHS = math.ceil(ITER_BUDGET / IT_PER_EPOCH)
    EPOCHS += EPOCHS % 2                                   # Train.py บันทึก epoch_last เฉพาะ epoch คู่
    RUN_NAME = f"udnet_n{N_SUBSET}_split{SEED}_seed{TRAIN_SEED}_it{ITER_BUDGET}"
else:
    EPOCHS = N_EPOCHS
    RUN_NAME = f"udnet_n{N_SUBSET}_split{SEED}_seed{TRAIN_SEED}_e{N_EPOCHS}"
print(f"{EPOCHS} epochs x {IT_PER_EPOCH} = {EPOCHS * IT_PER_EPOCH:,} iterations")

import os, sys, re, math, json, time, subprocess
import torch

GPU = torch.cuda.is_available()
GPU_NAME = torch.cuda.get_device_name(0) if GPU else "CPU"
THREADS = max(2, min(8, os.cpu_count() or 2))   # dataloader workers: runtime ที่ GPU แรงกว่ามักมี CPU มากกว่า
print("torch", torch.__version__, "| GPU:", GPU_NAME,
      f"({torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB)" if GPU else "", "| CPU cores:", os.cpu_count())
if not GPU:
    print(">>> ยังเป็น CPU: Runtime -> Change runtime type -> T4 GPU แล้วรันใหม่ตั้งแต่เซลล์นี้")

ROOT = "/content"
REPO = f"{ROOT}/UDnet"
UIEB = f"{ROOT}/uieb"        # ภาพที่แตกแล้ว (ดิสก์ของ runtime อ่านเร็ว)
DATA = f"{ROOT}/data"        # โครงสร้างโฟลเดอร์ที่สคริปต์ใช้
RESULTS = f"{ROOT}/results"  # ภาพผลลัพธ์
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/UDnet_work"   # ใช้โฟลเดอร์เดียวกันทั้งสอง notebook
else:
    WORK = f"{ROOT}/UDnet_work"
CACHE = f"{WORK}/downloads"
RUN_DIR = f"{WORK}/runs/{RUN_NAME}"
for d in (CACHE, RUN_DIR, UIEB, DATA, RESULTS):
    os.makedirs(d, exist_ok=True)
with open(f"{RUN_DIR}/env.txt", "a") as f:     # GPU ของแต่ละ session ใน run นี้ — ไว้เช็กว่าเทียบกันบน GPU แบบเดียวกัน
    f.write(f"{time.strftime('%Y-%m-%d %H:%M')}  {GPU_NAME}  torch {torch.__version__}  cpu {os.cpu_count()}\n")
print("run:", RUN_NAME, "\nไฟล์ถาวรเก็บที่:", WORK)

## §2 Clone repo

In [ ]:
NEED = ["Train.py", "Inference.py", "Evaluate.py", "CheckWeights.py", "colab_utils.py"]

if not os.path.isdir(f"{REPO}/.git"):
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "-q"], check=True)
os.chdir(REPO)
sys.path.insert(0, REPO)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)
missing = [f for f in NEED if not os.path.exists(f)]
assert not missing, f"repo ขาดไฟล์ {missing} — ต้อง push branch {BRANCH} ขึ้น GitHub ก่อน"

import importlib.util
if importlib.util.find_spec("gdown") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
import colab_utils as cu
print("พร้อม")

## §3 ข้อมูล

```
data/train/image        N_SUBSET ภาพดิบ จากส่วน train  <- Train.py (pseudo-label สร้างระหว่างเทรน)
data/test/input          90 ภาพดิบ                       <- Inference.py
data/test/reference      90 reference                    <- Evaluate.py
data/challenging-60      60 ภาพ ไม่มี reference
```
split 800/90 บันทึกที่ `UDnet_work/split_seed42.json` ครั้งเดียวแล้วใช้ซ้ำตลอด ถ้ามี split เดิมให้วางทับไฟล์นี้ก่อน (รูปแบบ `{"train": [...], "test": [...]}`)

In [ ]:
cu.prepare_uieb(UIEB, CACHE)
AUTHOR_W = cu.author_weights(CACHE)
split = cu.make_split(UIEB, f"{WORK}/split_seed{SEED}.json", SEED, 800)
cu.build_data_dirs(UIEB, DATA, split, N_SUBSET, SEED)

## §4 weight ของผู้แต่ง และภาพดิบ

`CheckWeights.py` ยืนยันว่า weight โหลดครบ (Inference.py ใช้ `strict=False` ถ้าไม่ครบจะไม่เตือน) แล้ววัดทุกอย่างด้วย `Evaluate.py` ตัวเดียวกัน

In [ ]:
cu.run(["CheckWeights.py", "--model", AUTHOR_W])
cu.run(["Inference.py", "--gpu_mode", str(GPU), "--threads", str(THREADS), "--input_dir", f"{DATA}/test/input/",
        "--output", f"{RESULTS}/", "--reference_out", "author", "--model", AUTHOR_W])
cu.run(["Evaluate.py", "--pred_dir", f"{RESULTS}/author", "--gt_dir", f"{DATA}/test/reference",
        "--csv", f"{WORK}/metrics_author.csv"])
cu.run(["Evaluate.py", "--pred_dir", f"{DATA}/test/input", "--gt_dir", f"{DATA}/test/reference",
        "--csv", f"{WORK}/metrics_raw.csv"])

## §5 เทรน UDnet ตั้งแต่ epoch 1

- `Train.py` ของ repo ตรงๆ แค่ใส่ `--start_iter 1` (ค่า default ของ repo คือ 250)
- แต่ละ run แยกโฟลเดอร์ `UDnet_work/runs/<RUN_NAME>/` รอบทดสอบจึงไม่ปนกับรอบจริง · หน้าจอแสดงแค่สรุป epoch, log เต็มอยู่ใน `train_log.txt`
- **runtime หลุด:** รัน §1–§3 ใหม่ แล้วรันเซลล์นี้ซ้ำ จะต่อจาก `epoch_last.pth` ล่าสุดให้เอง (optimizer เริ่มใหม่ เพราะ repo ไม่ได้เก็บ state ของ Adam)

In [ ]:
LOG = f"{RUN_DIR}/train_log.txt"
done = [int(m) for m in re.findall(r"Epoch (\d+) Complete", open(LOG).read())] if os.path.exists(LOG) else []
last_saved = (max(done) // 2 * 2) if done else 0          # epoch_last.pth is written on even epochs
start = last_saved + 1

args = ["Train.py", "--gpu_mode", str(GPU), "--data_dir", f"{DATA}/train/", "--data_train_dataset", "image",
        "--nEpochs", str(EPOCHS), "--batchSize", str(BATCH), "--threads", str(THREADS), "--seed", str(TRAIN_SEED),
        "--save_folder", f"{RUN_DIR}/", "--start_iter", str(start)]
if last_saved and os.path.exists(f"{RUN_DIR}/epoch_last.pth"):
    args += ["--resume_train", "True", "--model", f"{RUN_DIR}/epoch_last.pth"]
    print(f"ต่อจาก epoch {last_saved}")
elif last_saved:
    raise RuntimeError("log บอกว่าเคยเทรน แต่ไม่พบ epoch_last.pth — ลบ train_log.txt ถ้าจะเริ่มใหม่")

if start > EPOCHS:
    print(f"เทรนครบ {EPOCHS} epoch แล้ว")
else:
    cu.run_logged(args, LOG)
    print("เทรนเสร็จ")

## §6 ประเมิน weight ที่เทรนเอง + ตารางรวมทุก run

In [ ]:
OWN_W = f"{RUN_DIR}/epoch_{EPOCHS}.pth"
if not os.path.exists(OWN_W):
    OWN_W = f"{RUN_DIR}/epoch_last.pth"
print("ใช้", OWN_W)
cu.run(["CheckWeights.py", "--model", OWN_W])
cu.run(["Inference.py", "--gpu_mode", str(GPU), "--threads", str(THREADS), "--input_dir", f"{DATA}/test/input/",
        "--output", f"{RESULTS}/", "--reference_out", RUN_NAME, "--model", OWN_W])
cu.run(["Evaluate.py", "--pred_dir", f"{RESULTS}/{RUN_NAME}", "--gt_dir", f"{DATA}/test/reference",
        "--csv", f"{RUN_DIR}/metrics_test.csv"])
print()
cu.summarize(WORK)

## §7 (ทางเลือก) เทรนสั้นกว่านี้พอไหม

`Train.py` เก็บ `epoch_100.pth`, `epoch_200.pth`, ... ไว้ระหว่างเทรน เซลล์นี้วัดผลของแต่ละจุดบนชุดทดสอบ
ถ้าค่าที่ประมาณครึ่งทางใกล้กับค่าสุดท้าย run ที่เหลือในแผนลดจำนวน iteration ลงได้ (ต้องใช้ค่าเดียวกันทุก run รวมทั้งเส้นฐาน)

In [ ]:
import glob
snaps = sorted((int(re.search(r"epoch_(\d+)\.pth$", p).group(1)) * IT_PER_EPOCH, p)
               for p in glob.glob(f"{RUN_DIR}/epoch_*.pth") if re.search(r"epoch_\d+\.pth$", p))

def infer(ckpt, out):
    cu.run(["Inference.py", "--gpu_mode", str(GPU), "--threads", str(THREADS), "--input_dir", f"{DATA}/test/input/",
            "--output", os.path.dirname(out) + "/", "--reference_out", os.path.basename(out), "--model", ckpt], quiet=True)

cu.convergence(snaps, infer, f"{RESULTS}/{RUN_NAME}_conv", f"{DATA}/test/reference", RUN_DIR)